In [ ]:
import ee
import geemap
import os
import pandas as pd
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import json
from numpy.linalg import norm
from utils.cloudmask import apply_cloudmask


plt.rcParams['font.family'] = 'Times New Roman'

GEE_PROJECT_ID = "fusion-371234"
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)


# Study Area

In [ ]:
Map = geemap.Map(center=[-5.1919, -37.3424], zoom=12)
Map.add_basemap("HYBRID")

Map

In [ ]:
# Convert to GeoJSON format
geojson_dict = Map.user_roi.getInfo()
roi_name = "mossoro_roi"

# Save to file
os.makedirs("rois", exist_ok=True)
with open(f"rois/{roi_name}.json", "w") as f:
    json.dump(geojson_dict, f)

print(f"✅ Region saved as {roi_name}.json")

# Generate Time-Series

In [ ]:
S2_SR_COLLECTION_ID = "COPERNICUS/S2_HARMONIZED"

selected_bands = ['B1','B2','B3','B4','B5','B6', 'B7','B8','B8A','B9','B11','B12']

CLOUD_FILTER = 50 # 100 = No Filter, 0 = Full Filter
crs = "EPSG:4326"
scale = 10 # for Sentinel-2 is 10 meters resolution

year = 2017
year2 = 2024
start_date = f'{year}-01-01'
end_date   = f'{year2}-12-31'

roi_name = f"mossoro_roi"
with open(f"rois/{roi_name}.json", "r") as f:
    region_data = json.load(f)

region = ee.Geometry(region_data)
collection = (
    ee.ImageCollection(S2_SR_COLLECTION_ID)
    .filterDate(start_date, end_date)
    .filterBounds(region)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
    .select(selected_bands)
    .sort('system:time_start')
)

print("Number of images:", collection.size().getInfo())

os.makedirs(f"datasets/{year}_{roi_name}_time_series", exist_ok=True)

image_list = collection.toList(collection.size())
n_images = collection.size().getInfo()

for i in range(n_images):
    img = ee.Image(image_list.get(i))
    date = ee.Date(img.get('system:time_start')).format('YYYYMMdd').getInfo()
    filename = f"datasets/{year}_{roi_name}_time_series/S2_{date}.tif"
    geemap.ee_export_image(
        img.clip(region),
        filename=filename,
        scale=scale,
        region=region,
        crs=crs,
        file_per_band=False
    )
    print(f"✅ Exported {filename}")

In [ ]:
year = 2017
roi_name = "mossoro_roi"

apply_cloudmask(f"datasets/{year}_{roi_name}_time_series", 'cuda')

In [ ]:
import numpy as np
import rasterio
import torch
from pathlib import Path

folder = Path(f"datasets/{year}_{roi_name}_time_series")

device = "cuda" if torch.cuda.is_available() else "cpu"
SHADOW = torch.tensor([1, 2, 3], device=device)

for tif in sorted(folder.glob("*.tif")):
    with rasterio.open(tif) as src:
        mask = torch.from_numpy(src.read(src.count).astype(np.int32)).to(device)
    
    if torch.isin(mask, SHADOW).float().mean().item() > 0.05:
        tif.unlink()
        print(f"DELETED ✗ {tif.name}")
    else:
        print(f"KEPT    ✓ {tif.name}")

In [ ]:
path = f"datasets/{year}_{roi_name}_time_series"
scale = 3000

name_files = sorted(os.listdir(path))

rgb_maps = []
dates = []

for file in name_files:
    date = os.path.basename(file).split("_")[1].replace(".tif", "")
    dates.append(pd.to_datetime(date))

    with rasterio.open(os.path.join(path, file)) as src:
        red = src.read(4).astype(np.float32) / scale
        green = src.read(3).astype(np.float32) / scale
        blue = src.read(2).astype(np.float32) / scale

        nodata = src.nodata
        if nodata is not None:
            red = np.where(red == nodata, 0, red)
            green = np.where(green == nodata, 0, green)
            blue = np.where(blue == nodata, 0, blue)

        rgb = np.dstack([red, green, blue])
        rgb_maps.append(rgb)

rgb_maps = np.array(rgb_maps)

n_images = len(rgb_maps)
n_cols = 6
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4 * n_cols, 4 * n_rows),
    squeeze=False
)

for i, ax in enumerate(axes.flat):
    if i < n_images:
        ax.imshow(rgb_maps[i])
        ax.set_title(dates[i].strftime("%Y-%m-%d"))
        ax.axis("off")
    else:
        ax.axis("off")

plt.suptitle("RGB Time-Series Maps", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
from datetime import datetime
from utils.files_info import get_files_info

scale = 10000
years = [2017]

for year in years:

    print(f"\nProcessing {year}")

    ndvi_time_series = []
    dates = []

    path = f"datasets/{year}_mossoro_roi_time_series"
    n_files, name_files = get_files_info(path)

    file_date_pairs = []
    for file in name_files:
        date_str = os.path.basename(file).split("_")[1].replace(".tif", "")
        date = datetime.strptime(date_str, "%Y%m%d")
        file_date_pairs.append((file, date))

    file_date_pairs.sort(key=lambda x: x[1])

    print(f"Total scenes: {len(file_date_pairs)}")

    for file, date in file_date_pairs:

        print(f"  Reading {date.strftime('%Y-%m-%d')}")

        with rasterio.open(os.path.join(path, file)) as src:
            red = src.read(4).astype(np.float32) / scale
            nir = src.read(8).astype(np.float32) / scale

            nodata = src.nodata
            if nodata is not None:
                mask = (red == nodata) | (nir == nodata)
                red[mask] = np.nan
                nir[mask] = np.nan

            ndvi = np.where(
                (nir + red) != 0,
                (nir - red) / (nir + red),
                np.nan
            )

        ndvi_time_series.append(ndvi)
        dates.append(date)

    ndvi_cube = np.array(ndvi_time_series)
    T, H, W = ndvi_cube.shape
    print("NDVI cube:", ndvi_cube.shape)

    # (T, H, W) → (Npixels, T)
    pixels_ts = ndvi_cube.reshape(T, -1).T

    valid = ~np.all(np.isnan(pixels_ts), axis=1)
    pixels_ts = pixels_ts[valid]

    print("Valid pixels:", pixels_ts.shape[0])

    plt.figure(figsize=(11, 6))

    for ts in pixels_ts:
        plt.plot(dates, ts, color="green", alpha=0.05)

    plt.title(f"NDVI pixel time series — {year}")
    plt.xlabel("Date")
    plt.ylabel("NDVI")

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator())

    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    mean_ts = np.nanmean(pixels_ts, axis=0)
    median_ts = np.nanmedian(pixels_ts, axis=0)
    std_ts = np.nanstd(pixels_ts, axis=0)

    plt.figure(figsize=(11, 6))

    plt.plot(dates, mean_ts, label="Mean NDVI", color="black", linewidth=2)
    plt.plot(dates, median_ts, label="Median NDVI", color="red", linewidth=2)

    plt.fill_between(
        dates,
        mean_ts - std_ts,
        mean_ts + std_ts,
        alpha=0.3,
        label="±1σ (spatial variability)"
    )

    plt.title(f"NDVI temporal statistics — {year}")
    plt.xlabel("Date")
    plt.ylabel("NDVI")

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator())

    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()

    # plt.savefig(f"figures/ndvi_timeseries_{year}.png", dpi=200)
    plt.show()
